# CARFAC SAI on underwater audio

**Question:** what does a **stabilized auditory image (SAI)** look like on hydrophone recordings, and does it carry orca-call detection signal that a mel spectrogram does not?

Third in a line: [`carfac-vs-mel`](https://github.com/6cubed/216labs/tree/main/colabs/carfac-vs-mel) put NAP next to mel on ordinary audio, [`carfac-sai-drone`](https://github.com/6cubed/216labs/tree/main/colabs/carfac-sai-drone) added the SAI on voice buried in rotor noise. Here the audio is **underwater** and the framing is **detection**: passive acoustic monitoring (PAM) for marine mammals. Same lag conventions as the drone notebook, so the pictures are directly comparable.

| Frontend | What it is |
|----------|------------|
| **log-mel** | STFT → mel filterbank → dB. The de-facto PAM baseline (Orcasound's own Pod.Cast classifier uses 64-dim mel). |
| **CARFAC NAP** | Cochleagram: cascade of asymmetric resonators with fast-acting compression → neural activity pattern ([google/carfac](https://github.com/google/carfac)). |
| **CARFAC SAI** | The NAP run through **strobed temporal integration**: a running, sparse autocorrelation that stabilizes periodic structure into a still image with a **lag** axis. |

The SAI is the part with no mel equivalent. Mel throws away phase and fine timing inside a frame; the SAI keeps periodicity as an explicit lag dimension, which is why it is interesting for **pulsed** underwater sources (orca pulsed calls, echolocation click trains, propeller blade rate).

**Runtime:** CPU only. Sections 1–3 take ~2 minutes. Section 4 (the detection probe) is the slow part — the NumPy CARFAC runs at roughly **3–6× slower than real time**, so budget **~10–20 minutes** at the default size, and turn `N_PER_CLASS` down if you just want to see it work.

In [ ]:
# Install (Colab / fresh env). Safe to re-run. Needs Python >= 3.11.
%pip install -q "numpy" "scipy" "matplotlib" "librosa" "soundfile" "scikit-learn" \
  "carfac @ git+https://github.com/google/carfac.git@master#subdirectory=python"

In [ ]:
from __future__ import annotations

import csv
import os
import tarfile
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from scipy.signal import butter, sosfiltfilt
from IPython.display import Audio, display

import carfac.np.carfac as carfac_np
import carfac.sai as pysai

RNG = np.random.default_rng(216)

# --- audio / labels -------------------------------------------------------
FS = 20000          # native rate of the Orcasound Pod.Cast wavs
WINDOW_S = 2.45     # the Pod.Cast baseline decision window
HIGHPASS_HZ = 150.0 # standard PAM practice: strip ship/flow rumble
TARGET_RMS = 0.08   # per-window level normalisation (see note in section 2)

# --- mel ------------------------------------------------------------------
N_MELS, N_FFT, HOP = 64, 512, 128
FMIN, FMAX = 150.0, 10000.0

# --- carfac ---------------------------------------------------------------
MIN_POLE_HZ = 200.0     # lowest channel CF; see "tuning CARFAC for water" below
SAI_WIDTH = 400         # 20 ms of lag at 20 kHz
SAI_FUTURE_LAGS = 200   # half the image comes from after the trigger
SAI_TRIGGER_WINDOW = 800
SAI_HOP = 400           # -> one SAI frame every 20 ms (50 fps)

print("numpy", np.__version__, "| librosa", librosa.__version__)

## 1. An underwater dataset you can actually download

[**Orcasound Pod.Cast**](https://www.orcasound.net/portfolio/orcasound-lab-hydrophone) labelled test set, from the AWS Open Data bucket `s3://acoustic-sandbox` — **public, no credentials**.

- 21 WAV files (~61 s each, mono, 20 kHz) from the Orcasound Lab hydrophone, 27 Sep 2017
- `test.tsv` lists the **positive** intervals (Southern Resident killer whale calls); everything else is negative
- ~21 minutes total, roughly 4:1 negative:positive

One download, ~95 MB.

In [ ]:
URL = ("https://acoustic-sandbox.s3.amazonaws.com/labeled-data/detection/test/"
       "OrcasoundLab09272017_Test.tar.gz")
WORK = os.path.abspath("orcasound_podcast")
DATA_DIR = os.path.join(WORK, "OrcasoundLab09272017_Test")
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    tgz = os.path.join(WORK, "test.tar.gz")
    if not os.path.exists(tgz):
        print("downloading ~95 MB ...")
        urllib.request.urlretrieve(URL, tgz)
    with tarfile.open(tgz) as t:
        t.extractall(WORK)
    print("extracted")

wav_dir = os.path.join(DATA_DIR, "wav")
wavs = sorted(f for f in os.listdir(wav_dir) if f.endswith(".wav"))
info = sf.info(os.path.join(wav_dir, wavs[0]))
print(f"{len(wavs)} wav files | {info.samplerate} Hz | {info.frames/info.samplerate:.1f} s each")

In [ ]:
def load_positive_intervals(data_dir=DATA_DIR, min_dur=0.2):
    """Read test.tsv -> {wav_filename: [(start_s, duration_s), ...]}."""
    out = {}
    with open(os.path.join(data_dir, "test.tsv")) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            d = float(row["duration_s"])
            if d < min_dur:   # the TSV uses ~0 s rows to mean "no calls in this file"
                continue
            out.setdefault(row["wav_filename"], []).append((float(row["start_time_s"]), d))
    return out


def build_windows(data_dir=DATA_DIR, guard_s=0.5):
    """Positive = window centred on a labelled call. Negative = any window that
    misses every labelled call by at least guard_s."""
    pos = load_positive_intervals(data_dir)
    rows = []
    for name in sorted(f for f in os.listdir(os.path.join(data_dir, "wav")) if f.endswith(".wav")):
        meta = sf.info(os.path.join(data_dir, "wav", name))
        dur = meta.frames / meta.samplerate
        calls = pos.get(name, [])
        for start, length in calls:
            centre = start + length / 2.0
            w0 = min(max(centre - WINDOW_S / 2.0, 0.0), dur - WINDOW_S)
            rows.append({"wav": name, "start_s": round(w0, 3), "label": 1})
        t = 0.0
        while t + WINDOW_S <= dur:
            if not any(t < s + d + guard_s and s - guard_s < t + WINDOW_S for s, d in calls):
                rows.append({"wav": name, "start_s": round(t, 3), "label": 0})
            t += WINDOW_S
    return rows


windows = build_windows()
positives = [w for w in windows if w["label"] == 1]
negatives = [w for w in windows if w["label"] == 0]
print(f"{len(positives)} call windows, {len(negatives)} background windows, "
      f"{len(set(w['wav'] for w in windows))} source files")

## 2. The three frontends

**Tuning CARFAC for water.** CARFAC ships with a human cochlea: channels run from ~8.5 kHz down to a `min_pole_hz` of 30 Hz. Underwater that bottom end is a waste — it is all ship rumble, flow noise and mooring strum, and because it is *loud* it dominates CARFAC's AGC and the resulting SAI. Two changes:

1. a 150 Hz high-pass on the audio (routine in PAM), and
2. `min_pole_hz = 200`, which drops the noise-dominated channels.

SRKW calls sit at roughly 0.5–10 kHz, so nothing of interest is lost. (Echolocation clicks run to 80 kHz and are simply not in this 20 kHz recording — see the notes at the end.)

**Level.** Each window is RMS-normalised. Loudness alone separates calls from background reasonably well, and leaving it in would let every frontend score high for an uninteresting reason. Normalising forces the comparison onto spectro-temporal structure. Note this is not free: CARFAC's AGC is deliberately level-dependent, so we are removing an input it is designed to use.

**Lag bookkeeping.** `carfac.sai` puts the trigger at column `SAI_WIDTH - 1 - SAI_FUTURE_LAGS` with past lags to its left; we flip each frame so **lag increases to the right**, matching `carfac-sai-drone`. Zero is the trigger, negative is the look-ahead. A source repeating at *f* Hz puts ridges at 1/*f* and its multiples — 20 ms of lag here, so repetition rates down to 50 Hz.

In [ ]:
_HP_SOS = butter(4, HIGHPASS_HZ, btype="highpass", fs=FS, output="sos")


def read_window(name, start_s, data_dir=DATA_DIR):
    """Load one WINDOW_S window: DC-remove, high-pass, RMS-normalise."""
    y, fs = sf.read(os.path.join(data_dir, "wav", name),
                    start=int(round(start_s * FS)),
                    frames=int(round(WINDOW_S * FS)), dtype="float32")
    assert fs == FS, fs
    y = y - float(np.mean(y))
    y = sosfiltfilt(_HP_SOS, y)
    y = y / (float(np.sqrt(np.mean(y ** 2))) + 1e-9) * TARGET_RMS
    return y.astype(np.float32)


def mel_db(y):
    m = librosa.feature.melspectrogram(y=y, sr=FS, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
    return librosa.power_to_db(m, ref=1.0)


def design_underwater_carfac(fs=FS, min_pole_hz=MIN_POLE_HZ):
    cfp = carfac_np.design_carfac(n_ears=1, fs=float(fs),
                                  car_params=carfac_np.CarParams(min_pole_hz=min_pole_hz))
    return carfac_np.carfac_init(cfp)


def carfac_nap(y):
    """-> (n_samples, n_channels) NAP and the channel centre frequencies."""
    cfp = design_underwater_carfac()
    naps, cfp, _bm, _ohc, _agc = carfac_np.run_segment(cfp, y.astype(np.float64).reshape(-1, 1))
    if naps.ndim == 3:
        naps = naps[:, :, 0]
    return naps.astype(np.float32), np.asarray(cfp.pole_freqs, dtype=float).reshape(-1)


def carfac_sai(nap):
    """-> (n_frames, n_channels, SAI_WIDTH) SAI frames, lag increasing to the right."""
    params = pysai.SAIParams(num_channels=int(nap.shape[1]),
                             sai_width=SAI_WIDTH,
                             future_lags=SAI_FUTURE_LAGS,
                             num_triggers_per_frame=2,
                             trigger_window_width=SAI_TRIGGER_WINDOW,
                             input_segment_width=SAI_HOP)
    sai = pysai.SAI(params)
    x = np.ascontiguousarray(nap.T, dtype=np.float64)   # SAI wants (channels, samples)
    frames = [sai.RunSegment(x[:, s:s + SAI_HOP])[:, ::-1].copy()
              for s in range(0, x.shape[1] - SAI_HOP + 1, SAI_HOP)]
    return np.stack(frames).astype(np.float32)


def nap_cochleagram(nap, frame=N_FFT, hop=HOP):
    """Frame the sample-rate NAP down to the mel time grid for plotting."""
    n = nap.shape[0]
    cols = [np.mean(np.abs(nap[s:s + frame]), axis=0)
            for s in range(0, max(n - frame + 1, 1), hop)]
    return np.stack(cols, axis=1)


def log_compress(x, floor=0.2):
    return np.log1p(np.maximum(x, 0.0) / floor)


def lag_axis_ms():
    """Lag axis (ms) for the flipped frames: 0 = trigger, negative = look-ahead."""
    return (np.arange(SAI_WIDTH) - SAI_FUTURE_LAGS) / FS * 1000.0


def channel_yticks(ax, pole_freqs, n=5):
    """Label a flipped (low-CF-at-bottom) channel axis with centre frequencies."""
    cfs = np.asarray(pole_freqs)[::-1]
    pos = np.linspace(0, len(cfs) - 1, n).astype(int)
    ax.set_yticks(pos + 0.5)
    ax.set_yticklabels([f"{cfs[p]:.0f}" for p in pos])
    ax.set_ylabel("channel CF (Hz)")


_cfp = design_underwater_carfac()
print(f"CARFAC: {_cfp.n_ch} channels, CF {_cfp.pole_freqs.min():.0f}-{_cfp.pole_freqs.max():.0f} Hz")
print(f"SAI frame: {_cfp.n_ch} x {SAI_WIDTH}, lag axis "
      f"{lag_axis_ms()[0]:.1f}..{lag_axis_ms()[-1]:.1f} ms, {FS/SAI_HOP:.0f} frames/s")

## 3. What it looks like: one call window, one background window

In [ ]:
pool_pos, pool_neg = list(positives), list(negatives)
RNG.shuffle(pool_pos)
RNG.shuffle(pool_neg)
picks = [("orca call (SRKW)", pool_pos[0]), ("background", pool_neg[0])]

looks = []
for title, w in picks:
    y = read_window(w["wav"], w["start_s"])
    t0 = time.time()
    nap, pole_freqs = carfac_nap(y)
    sai = carfac_sai(nap)
    print(f"{title:18s} {w['wav']} @ {w['start_s']:.1f}s  "
          f"({time.time()-t0:.1f}s of compute for {WINDOW_S:.2f}s of audio)")
    looks.append({"title": title, "w": w, "y": y, "nap": nap, "sai": sai, "cf": pole_freqs})
    display(Audio(y, rate=FS))

In [ ]:
lag_ms = lag_axis_ms()
fig, axes = plt.subplots(2, 4, figsize=(20, 8), constrained_layout=True)

for r, item in enumerate(looks):
    y, nap, sai = item["y"], item["nap"], item["sai"]
    n_ch = nap.shape[1]

    ax = axes[r][0]
    ax.plot(np.arange(len(y)) / FS, y, lw=0.4, color="#222")
    ax.set_xlim(0, WINDOW_S)
    ax.set_xlabel("time (s)")
    ax.set_title(f"{item['title']}\n{item['w']['wav']} @ {item['w']['start_s']:.1f}s", fontsize=10)

    ax = axes[r][1]
    img = librosa.display.specshow(mel_db(y), sr=FS, hop_length=HOP, x_axis="time",
                                   y_axis="mel", fmin=FMIN, fmax=FMAX, ax=ax, cmap="magma")
    ax.set_title(f"log-mel ({N_MELS} bands)", fontsize=10)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

    ax = axes[r][2]
    coch = log_compress(nap_cochleagram(nap))
    img = ax.imshow(coch[::-1], aspect="auto", origin="lower", cmap="magma",
                    extent=[0, WINDOW_S, 0, n_ch])
    ax.set_xlabel("time (s)")
    channel_yticks(ax, item["cf"])
    ax.set_title(f"CARFAC NAP cochleagram ({n_ch} ch)", fontsize=10)
    fig.colorbar(img, ax=ax)

    ax = axes[r][3]
    sai_img = log_compress(sai.mean(axis=0))
    img = ax.imshow(sai_img[::-1], aspect="auto", origin="lower", cmap="magma",
                    extent=[lag_ms[0], lag_ms[-1], 0, n_ch])
    ax.axvline(0.0, color="w", lw=0.6, alpha=0.6)
    ax.set_xlabel("lag (ms)")
    channel_yticks(ax, item["cf"])
    ax.set_title(f"CARFAC SAI, mean of {len(sai)} frames", fontsize=10)
    fig.colorbar(img, ax=ax)

plt.show()

In [ ]:
# The SAI is really a movie, not a still. Six frames across the call window.
# The first frame is nearly empty: the SAI input buffer has not filled yet and
# CARFAC's AGC is still settling, so ignore the first ~0.5 s of any run.
sai = looks[0]["sai"]
idx = np.linspace(0, len(sai) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(20, 3.6), constrained_layout=True)
for ax, i in zip(axes, idx):
    ax.imshow(log_compress(sai[i])[::-1], aspect="auto", origin="lower", cmap="magma",
              extent=[lag_ms[0], lag_ms[-1], 0, sai.shape[1]])
    ax.set_title(f"t = {i * SAI_HOP / FS:.2f}s", fontsize=9)
    ax.set_xlabel("lag (ms)")
fig.suptitle(f"SAI frames — {looks[0]['title']}", fontsize=11)
plt.show()

In [ ]:
# Collapsing the channel axis instead of the time axis gives a "lag over time"
# view: periodicity as it evolves through the window. This keeps the time
# structure that the mean-SAI image throws away.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), constrained_layout=True)
for ax, item in zip(axes, looks):
    lag_over_time = log_compress(item["sai"].mean(axis=1)).T   # (lag, frame)
    img = ax.imshow(lag_over_time, aspect="auto", origin="lower", cmap="magma",
                    extent=[0, WINDOW_S, lag_ms[0], lag_ms[-1]])
    ax.axhline(0.0, color="w", lw=0.5, alpha=0.5)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("lag (ms)")
    ax.set_title(f"SAI lag x time — {item['title']}", fontsize=10)
    fig.colorbar(img, ax=ax)
plt.show()

### Reading these

The mel panel is the familiar one: the call shows up as tonal contours around 1–2 kHz with harmonics; the background window is broadband noise with the odd impulsive tick.

The mean-SAI panels look, at first glance, **almost identical between the two classes**, and both are dominated by the bright ridge at lag 0 with a fan of rings spreading out from it. That fan is largely the filterbank's *own* impulse response — each channel rings at its CF, so any transient produces a chirp-shaped ridge regardless of what made it. Averaged over 2.45 s, that self-structure swamps the class difference.

So: **a time-averaged SAI is not a drop-in "better spectrogram" you can eyeball.** Its class structure shows up in three other places — frame by frame (the movie), in the lag × time view (where the call's periodicity comes and goes while background ringing stays flat), and in the *difference* between class means, which section 4 plots.

## 4. Does the SAI carry detection signal?

A small, honest probe — **not** a benchmark.

For each window we build a fixed-size image from each frontend, mean-pool it to 32×32, flatten, and fit a logistic regression. Cross-validation is **grouped by source WAV file**, so no model is ever tested on the same recording it trained on. Score is ROC-AUC.

Four feature sets, chosen so the ablation is meaningful:

| | image | question it answers |
|---|---|---|
| **mel** | frequency × time | the baseline |
| **NAP** | channel × time | does the nonlinear filterbank + AGC help on its own? |
| **SAI (mean)** | channel × lag | does the stabilized image, averaged over the window, separate the classes? |
| **SAI (lag×time)** | lag × time | same SAI frames, but collapsing *channels* instead of *time* — periodicity as it evolves |

The last two are the same SAI frames marginalised along different axes, which is the point: it separates "SAI as a still image" from "SAI as a time series".

`N_PER_CLASS = 24` is ~2 minutes of audio and takes roughly 10–20 minutes on Colab CPU, almost all of it inside CARFAC's per-sample Python loop. Drop it to 6 for a quick smoke test.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_PER_CLASS = 24   # <- turn this down for a fast run


def resize_mean(x, shape=(32, 32)):
    rows = np.stack([b.mean(axis=0) for b in np.array_split(x, shape[0], axis=0)])
    return np.stack([b.mean(axis=1) for b in np.array_split(rows, shape[1], axis=1)], axis=1)


def image_features(img, shape=(32, 32)):
    return resize_mean(np.asarray(img, dtype=np.float64), shape).ravel()


subset = pool_pos[:N_PER_CLASS] + pool_neg[:N_PER_CLASS]
print(f"{len(subset)} windows = {len(subset) * WINDOW_S:.0f}s of audio")

FRONTENDS = ("mel", "NAP", "SAI (mean)", "SAI (lag x time)")
feats = {k: [] for k in FRONTENDS}
sai_images, labels, groups = [], [], []
t0 = time.time()
for i, w in enumerate(subset):
    y = read_window(w["wav"], w["start_s"])
    nap, _ = carfac_nap(y)
    sai = carfac_sai(nap)
    sai_img = log_compress(sai.mean(axis=0))              # channel x lag
    sai_lag_time = log_compress(sai.mean(axis=1)).T       # lag x time
    feats["mel"].append(image_features(mel_db(y)))
    feats["NAP"].append(image_features(log_compress(nap_cochleagram(nap))))
    feats["SAI (mean)"].append(image_features(sai_img))
    feats["SAI (lag x time)"].append(image_features(sai_lag_time))
    sai_images.append(sai_img)
    labels.append(w["label"])
    groups.append(w["wav"])
    done = time.time() - t0
    print(f"  {i+1}/{len(subset)}  {done:5.0f}s elapsed, ~{done/(i+1)*(len(subset)-i-1):5.0f}s left",
          end="\r", flush=True)

feats = {k: np.asarray(v) for k, v in feats.items()}
sai_images = np.asarray(sai_images)
labels = np.asarray(labels)
groups = np.asarray(groups)
print(f"\ndone in {time.time()-t0:.0f}s | {labels.sum()} calls, {(1-labels).sum()} background, "
      f"{len(set(groups))} files")

In [ ]:
def grouped_auc(X, n_splits=5, C=0.05):
    X = np.asarray(X, dtype=np.float64)
    scores = []
    for train, test in GroupKFold(n_splits=n_splits).split(X, labels, groups):
        if len(np.unique(labels[test])) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=C))
        clf.fit(X[train], labels[train])
        scores.append(roc_auc_score(labels[test], clf.predict_proba(X[test])[:, 1]))
    return float(np.mean(scores)), float(np.std(scores)), len(scores)


print(f"{'frontend':<18}{'dim':>6}{'AUC':>10}{'sd':>8}{'folds':>7}")
for name in FRONTENDS:
    mean, sd, k = grouped_auc(feats[name])
    print(f"{name:<18}{feats[name].shape[1]:>6}{mean:>10.3f}{sd:>8.3f}{k:>7}")

combo = np.hstack([feats["mel"], feats["SAI (lag x time)"]])
mean, sd, k = grouped_auc(combo)
print(f"{'mel + SAI(lag x t)':<18}{combo.shape[1]:>6}{mean:>10.3f}{sd:>8.3f}{k:>7}")

In [ ]:
# What the SAI says about the two classes, once the filterbank's own ringing
# is subtracted out by differencing the class means.
call_mean = sai_images[labels == 1].mean(axis=0)
bg_mean = sai_images[labels == 0].mean(axis=0)
diff = call_mean - bg_mean
n_ch = call_mean.shape[0]
extent = [lag_ms[0], lag_ms[-1], 0, n_ch]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6), constrained_layout=True)
for ax, (title, image, cmap) in zip(axes, [
    (f"mean SAI — calls (n={int(labels.sum())})", call_mean, "magma"),
    (f"mean SAI — background (n={int((1-labels).sum())})", bg_mean, "magma"),
    ("calls − background", diff, "RdBu_r"),
]):
    kw = {}
    if cmap == "RdBu_r":
        v = float(np.max(np.abs(image)))
        kw = {"vmin": -v, "vmax": v}
    img = ax.imshow(image[::-1], aspect="auto", origin="lower", cmap=cmap, extent=extent, **kw)
    ax.axvline(0.0, color="k" if cmap == "RdBu_r" else "w", lw=0.5, alpha=0.4)
    ax.set_xlabel("lag (ms)")
    channel_yticks(ax, looks[0]["cf"])
    ax.set_title(title, fontsize=10)
    fig.colorbar(img, ax=ax)
plt.show()

## 5. What we saw, and what to do next

One run at `N_PER_CLASS = 24` (48 windows, 16 source files, 3 usable grouped folds):

| frontend | AUC |
|---|---|
| mel | 0.98 ± 0.03 |
| CARFAC NAP | 0.94 ± 0.08 |
| CARFAC SAI (mean over window) | 0.69 ± 0.24 |
| CARFAC SAI (lag × time) | 0.82 ± 0.10 |

Three things fall out of that:

1. **Mel is hard to beat here, and the cochleagram nearly matches it.** CARFAC's filterbank and AGC are not the problem.
2. **Averaging the SAI over the window is what costs you.** A 2.45 s decision window contains a call lasting a fraction of a second; collapsing 122 SAI frames into one image discards exactly the temporal envelope that makes the call detectable. Marginalising over channels instead — keeping lag × time — recovers most of the gap.
3. **The SAI's distinctive cue has nothing to bite on in this recording.** Lag structure earns its keep on repetitive pulse trains, and the one signal class with strong pulse-rate structure (echolocation clicks) is above the 10 kHz Nyquist of these files.

**Caveats, up front.** Tens of windows from a *single day at a single hydrophone* is not enough to rank frontends. Grouped CV stops the most obvious leak (train and test never share a recording) but every window still comes from the same node, the same sea state and the same instrument. The error bars overlap. Treat the table as a smoke test for "is there signal here at all", nothing more.

**Where CARFAC could plausibly earn its keep underwater.** Not as a nicer-looking spectrogram — as a *timing* representation:

- **Click trains.** Odontocete echolocation is a pulse train, and inter-click interval is the identifying feature. That is a lag-axis quantity, exactly what the SAI represents natively and what mel destroys inside its frames.
- **Pulsed calls.** SRKW pulsed calls have a pulse repetition rate that reads directly as a lag ridge.
- **Vessel noise.** Propeller blade rate is periodic; cavitation is not. Shaft rate shows up as a lag, not a frequency band.
- **Low SNR.** CARFAC's AGC is a per-channel adaptive gain, which is a more principled answer to a non-stationary ocean noise floor than a global dB normalisation.

**Known limits of this setup.**

- 20 kHz sample rate caps us at 10 kHz. Echolocation clicks run to 80 kHz, so the single most SAI-friendly signal class is *absent from this recording*. Re-running on a high-rate dataset (MBARI's `pacific-sound-256khz`, or NOAA's PAM archive on GCS) is the obvious next step.
- `SAI_WIDTH = 400` gives a 20 ms lag window — repetition rates down to ~50 Hz. Slower rhythms need a wider SAI.
- The NumPy CARFAC is a per-sample Python loop and is the entire runtime cost. `carfac.jax` exists and is far faster; porting the probe to it is what makes a real-sized experiment (thousands of windows) feasible.
- Flattened 32×32 pooled images plus logistic regression is a deliberately weak read-out. It measures "is the class difference linearly available", not "how good can this frontend be".

**If this thread continues:** high-rate click data, `carfac.jax` for throughput, and a comparison against the actual Pod.Cast baseline model rather than a linear probe.

### Credits

Audio: [Orcasound](https://www.orcasound.net/) Pod.Cast labelled data, AWS Open Data (`s3://acoustic-sandbox`), CC-BY. Model: [google/carfac](https://github.com/google/carfac) (Apache 2.0), Dick Lyon, *Human and Machine Hearing*.